# Notebook 04: Model Training & Leakage Prevention
## Smart CCTV AI Safety & Identification Platform

This notebook covers:
1. Group-aware Subject Split (preventing identical-frame training/testing leakage)
2. Training Decision Tree Classifier
3. Training Random Forest Ensemble Classifier
4. Cross-validation and Hyperparameter tuning
5. Exporting trained model weights (`fall_detection_model.pkl`)

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle
import json
from src.features.feature_extractor import FEATURE_COLUMNS

df = pd.read_csv('../data/raw/fall_activity_dataset.csv')
print(f"Total rows: {len(df)}")

In [ ]:
# Group-aware Subject Split
unique_subjects = list(df['subject_id'].unique())
train_subs, test_subs = train_test_split(unique_subjects, test_size=0.20, random_state=42)

train_df = df[df['subject_id'].isin(train_subs)]
test_df = df[df['subject_id'].isin(test_subs)]

X_train = np.array(train_df[FEATURE_COLUMNS], dtype=np.float32)
y_train = np.array(train_df['label'])
X_test = np.array(test_df[FEATURE_COLUMNS], dtype=np.float32)
y_test = np.array(test_df['label'])

print(f"Train set: {len(X_train)} samples ({len(train_subs)} subjects)")
print(f"Test set:  {len(X_test)} samples ({len(test_subs)} subjects)")

In [ ]:
# Train Decision Tree & Random Forest
dt = DecisionTreeClassifier(max_depth=6, random_state=42)
dt.fit(X_train, y_train)
print(f"Decision Tree Test Accuracy: {dt.score(X_test, y_test)*100:.2f}%")

rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print(f"Random Forest Test Accuracy: {rf.score(X_test, y_test)*100:.2f}%")

In [ ]:
# Feature Importance Analysis
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=False)
print("=== Random Forest Feature Importances ===")
print(importances)